In [0]:
display(spark.sql("""
        SELECT
          COUNT(*) AS linhas,
          SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END) AS partida_real_true_null,
          SUM(CASE WHEN partida_real = "null" THEN 1 ELSE 0 END) AS partida_real_string_null,
          SUM(CASE WHEN partida_prevista = "null" THEN 1 ELSE 0 END) AS partida_prevista_string_null,
          SUM(CASE WHEN partida_prevista LIKE '%.%' THEN 1 ELSE 0 END) AS with_fraction_of_second
        FROM aviation.bronze.vra
                  """))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS aviation.silver")
spark.sql("""
CREATE OR REPLACE TABLE aviation.silver.vra AS
WITH typed AS (
    SELECT
      icao_empresa_aerea,
      numero_voo,
      codigo_autorizacao_di,
      codigo_tipo_linha,
      icao_aerodromo_origem,
      icao_aerodromo_destino,
      try_cast(nullif(partida_prevista, "null") AS TIMESTAMP) AS partida_prevista,
      try_cast(nullif(partida_real, "null") AS TIMESTAMP) AS partida_real,
      try_cast(nullif(chegada_prevista, "null") AS TIMESTAMP) AS chegada_prevista,
      try_cast(nullif(chegada_real, "null") AS TIMESTAMP) AS chegada_real,
      situacao_voo,
      nullif(codigo_justificativa, "N/A") AS codigo_justificativa,
      _arquivo_origem,
      _ingerido_em
    FROM aviation.bronze.vra
)
SELECT
  icao_empresa_aerea,
  numero_voo,
  codigo_autorizacao_di,
  codigo_tipo_linha,
  icao_aerodromo_origem,
  icao_aerodromo_destino,

  partida_prevista,
  CAST(partida_prevista AS DATE) AS partida_prevista_date,
  date_format(partida_prevista, "HH:mm") AS partida_prevista_hora,

  partida_real,
  CAST(partida_real AS DATE) AS partida_real_date,
  date_format(partida_real, "HH:mm") AS partida_real_hora,

  chegada_prevista,
  CAST(chegada_prevista AS DATE) AS chegada_prevista_date,
  date_format(chegada_prevista, "HH:mm") AS chegada_prevista_hora,

  chegada_real,
  CAST(chegada_real AS DATE) AS chegada_real_date,
  date_format(chegada_real, "HH:mm") AS chegada_real_hora,
  
  situacao_voo,
  codigo_justificativa,

  CAST(timestampdiff(MINUTE, partida_prevista, partida_real) AS INT) AS partida_delay,
  CAST(timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS chegada_delay,
  CAST(timestampdiff(MINUTE, partida_prevista, partida_real) - timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS minutes_recovered,

  _arquivo_origem,
  _ingerido_em,
  current_timestamp() AS _processed_on
FROM typed
""")


In [0]:
display(spark.sql("""
SELECT
  (SELECT COUNT(*) FROM aviation.bronze.vra) AS bronze,
  (SELECT COUNT(*) FROM aviation.silver.vra) AS silver,
  (SELECT COUNT(*) FROM aviation.bronze.vra)
    - (SELECT COUNT(*) FROM aviation.silver.vra) AS diff
"""))

In [0]:
display(spark.sql("""
SELECT
  COUNT_IF(partida_prevista IS NULL) AS partida_prevista_null,
  COUNT_IF(partida_real IS NULL) AS partida_real_null,
  COUNT_IF(chegada_prevista IS NULL) AS chegada_prevista_null,
  COUNT_IF(chegada_real IS NULL) AS chegada_real_null
FROM aviation.silver.vra
"""))

In [0]:
display(spark.sql("""
SELECT
  COUNT_IF(partida_prevista IS NOT NULL AND partida_prevista <> 'null'
           AND try_cast(partida_prevista AS TIMESTAMP) IS NULL) AS partida_prevista_invalid,
  COUNT_IF(partida_real IS NOT NULL AND partida_real <> 'null'
           AND try_cast(partida_real AS TIMESTAMP) IS NULL) AS partida_real_invalid,
  COUNT_IF(chegada_prevista IS NOT NULL AND chegada_prevista <> 'null'
           AND try_cast(chegada_prevista AS TIMESTAMP) IS NULL) AS chegada_prevista_invalid,
  COUNT_IF(chegada_real IS NOT NULL AND chegada_real <> 'null'
           AND try_cast(chegada_real AS TIMESTAMP) IS NULL) AS chegada_real_invalid
FROM aviation.bronze.vra
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE aviation.silver.companies AS
SELECT
  icao, sigla_iata, razao_social, servico, cidade, uf, situacao,
  'national' AS registration_source,
  _arquivo_origem, _ingerido_em,
  current_timestamp() AS _processed_on
FROM aviation.bronze.national_companies

UNION ALL

SELECT
  icao, sigla_iata, razao_social, servico, cidade, uf, situacao,
  'foreign' AS registration_source,
  _arquivo_origem, _ingerido_em,
  current_timestamp() AS _processed_on
FROM aviation.bronze.foreign_companies
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE aviation.silver.airfield AS
SELECT
  icao,
  ciad,
  nome,
  municipio,
  uf AS uf_name,
  municipio_servido,
  uf_servido AS served_uf_name,
  latitude AS latitude_dms,
  longitude AS longitude_dms,
  try_cast(replace(altitude, ',', '.') AS DOUBLE) AS altitude_m,
  situacao,
  _ingerido_em,
  current_timestamp() AS _processed_on
FROM aviation.bronze.airfield
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE aviation.silver.operation_codes AS
SELECT
  domain,
  code,
  description,
  current_timestamp() AS _processed_on
FROM aviation.bronze.operation_codes
""")

In [0]:
table_comments = {
    "vra": "One row per flight leg from bronze.vra, with typed timestamps and delay calculations.",
    "companies": "National and foreign company registries combined without removing records. registration_source identifies the source.",
    "airfield": "Mirror of the ANAC public airfield registry. UF fields contain state names, not abbreviations.",
    "operation_codes": "Reference descriptions for flight authorization and route type codes.",
}

for table, comment in table_comments.items():
    spark.sql(f"COMMENT ON TABLE aviation.silver.{table} IS '{comment}'")

In [0]:
column_comments = {
    ("vra", "partida_delay"): "Actual departure minus scheduled departure, in minutes. Negative values mean an early departure.",
    ("vra", "chegada_delay"): "Actual arrival minus scheduled arrival, in minutes. Negative values mean an early arrival.",
    ("vra", "minutes_recovered"): "Departure delay minus arrival delay, in minutes.",
    ("companies", "registration_source"): "Registry that supplied this record: national or foreign.",
    ("airfield", "uf_name"): "State name written in full, not a two-letter abbreviation.",
    ("airfield", "served_uf_name"): "Served state name written in full.",
    ("airfield", "altitude_m"): "Airfield altitude in metres, converted from a comma-decimal source value.",
    ("airfield", "latitude_dms"): "Latitude as supplied by ANAC in degrees, minutes and seconds.",
    ("airfield", "longitude_dms"): "Longitude as supplied by ANAC in degrees, minutes and seconds.",
    ("operation_codes", "domain"): "Code family; use with code to identify a reference value.",
    ("operation_codes", "code"): "Operation code within its domain.",
}

for (table, column), comment in column_comments.items():
    if comment is not None:
        spark.sql(
            f"ALTER TABLE aviation.silver.{table} "
            f"ALTER COLUMN {column} COMMENT '{comment}'"
        )

In [0]:
display(spark.sql("""
SELECT 'companies' AS silver_table,
       (SELECT COUNT(*) FROM aviation.bronze.national_companies)
       + (SELECT COUNT(*) FROM aviation.bronze.foreign_companies) AS expected_rows,
       (SELECT COUNT(*) FROM aviation.silver.companies) AS actual_rows
UNION ALL
SELECT 'airfield',
       (SELECT COUNT(*) FROM aviation.bronze.airfield),
       (SELECT COUNT(*) FROM aviation.silver.airfield)
UNION ALL
SELECT 'operation_codes',
       (SELECT COUNT(*) FROM aviation.bronze.operation_codes),
       (SELECT COUNT(*) FROM aviation.silver.operation_codes)
"""))